# Widgets & Configuration

In [0]:
import pandas as pd
from pyspark.sql.functions import (
    col, count, when, isnull, trim, length, countDistinct, 
    min, max, avg, round as spark_round, desc
)

# Setup widgets for environment-agnostic paths
dbutils.widgets.text("project_catalog", "vstone_catalog", "1. Catalog Name")
dbutils.widgets.text("raw_schema", "raw", "2. Raw Schema")
dbutils.widgets.text("landing_volume", "landing", "3. Landing Volume")

# Fetch values into variables
CATALOG = dbutils.widgets.get("project_catalog")
RAW_SCHEMA = dbutils.widgets.get("raw_schema")
LANDING_VOL = dbutils.widgets.get("landing_volume")

# Construct the dynamic landing path using Unity Catalog syntax
LANDING_PATH = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/{LANDING_VOL}"

# File Manifest with your original format and options
FILES = {
    "1_main": {"path": f"{LANDING_PATH}/1_main.csv", "format": "csv", "opts": {"sep": ","}},
    "catalogs": {"path": f"{LANDING_PATH}/catalogs.csv", "format": "csv", "opts": {"sep": ";"}},
    "geolocation": {"path": f"{LANDING_PATH}/final_geografic.csv", "format": "csv", "opts": {"sep": ","}},
    "text": {"path": f"{LANDING_PATH}/1_text.csv", "format": "csv", "opts": {"sep": ",", "multiLine": "true", "escape": '"'}},
    "photos": {"path": f"{LANDING_PATH}/1_photo.csv", "format": "csv", "opts": {"sep": ","}},
}

# DATA PROFILING & QUALITY AUDIT LOGIC

In [0]:

def profile_file(name, path, fmt, options):
    print(f"\n{'='*95}\n DATA AUDIT REPORT: {name.upper()}\n{'='*95}")
    
    # FIX: inferSchema=False use kar rahe hain taaki '₽' ya Cyrillic text BIGINT mein cast na ho
    df = (spark.read
          .option("header", True)
          .option("inferSchema", False) # Sab columns ko string treat karega initially
          .option("encoding", "UTF-8")
          .options(**options)
          .format(fmt)
          .load(path))
    
    total_rows = df.count()
    if total_rows == 0: 
        print(f"⚠️ Warning: File {name} is empty.")
        return None, 0

    # 2. COMPLETE METRICS AUDIT (Using string-safe aggregations)
    quality_exprs = []
    for c in df.columns:
        quality_exprs.extend([
            count(when(isnull(col(c)) | (trim(col(c)) == ""), c)).alias(f"{c}_nulls"),
            countDistinct(col(c)).alias(f"{c}_distinct")
            # Note: Zero count hata diya hai kyunki inferSchema=False mein sab string hai
        ])
    
    audit_results = df.select(quality_exprs).toPandas().transpose()
    audit_results.columns = ["Value"]
    
    profile_rows = []
    for c in df.columns:
        null_count = audit_results.loc[f"{c}_nulls", "Value"]
        distinct_count = audit_results.loc[f"{c}_distinct", "Value"]
        
        profile_rows.append({
            "Column_Name": c,
            "Data_Type": dict(df.dtypes)[c],
            "Total_Count": total_rows,
            "Null_Count": null_count,
            "Null_Percentage": round((null_count / total_rows) * 100, 2),
            "Distinct_Values": distinct_count,
            "Completeness": f"{round(100 - (null_count/total_rows*100), 2)}%"
        })
    
    pdf_final = pd.DataFrame(profile_rows)
    print(f" Dataset Stats: {total_rows:,} Rows | {len(df.columns)} Columns")
    display(pdf_final.style.background_gradient(cmap='YlOrRd', subset=['Null_Percentage']))

    # 3. SMART INTEGRITY CHECK
    if name == "catalogs":
        pk_cols = ["Марка", "Модель", "Поколение", "Комплектация"]
        distinct_rows = df.select(pk_cols).distinct().count()
        pk_candidate = "Composite (Catalog Schema)"
    else:
        # Safety for id or first column
        pk_candidate = "id" if "id" in df.columns else df.columns[0]
        distinct_rows = df.select(pk_candidate).distinct().count()
    
    duplicate_count = total_rows - distinct_rows
    
    integrity_summary = pd.DataFrame([
        {"Metric": "Primary Key Candidate", "Status": pk_candidate},
        {"Metric": "Duplicate Records", "Status": f" {duplicate_count:,}" if duplicate_count > 0 else " 0 Duplicates"},
        {"Metric": "Uniqueness Ratio", "Status": f"{round((distinct_rows/total_rows)*100, 2)}%"}
    ])
    display(integrity_summary)
    
    return df, total_rows

# --- AUTOMATED AUDIT EXECUTION ---
profiled = {}
for name, cfg in FILES.items():
    try:
        df, rows = profile_file(name, cfg["path"], cfg["format"], cfg["opts"])
        profiled[name] = {"df": df}
    except Exception as e:
        # Error handling for specific files
        print(f" Error profiling {name}: {str(e)}")

# --- 1_MAIN KPI SUMMARY (Manual Casting for specific analysis) ---
if "1_main" in profiled:
    print("\n" + " " * 30 + " 1_MAIN BUSINESS LOGIC KPIs " + " " * 30)
    # Analysis ke waqt 'cost' ko manually cast karenge taaki error na aaye
    df_main = profiled["1_main"]["df"]
    # ₽ symbol filter karke convert kar rahe hain
    df_clean = df_main.withColumn("cost_num", col("cost").cast("double"))
    
    display(df_clean.select(
        spark_round(avg("cost_num"), 0).alias("Avg_Price_RUB"),
        countDistinct("marka").alias("Total_Brands")
    ))